# 🤖 Umělá inteligence – Student Lab Workbook (Reading Week)

**Téma:** Teorie her, Nashova rovnováha, Algoritmus Minimax a Evoluční dynamika populací

**Cílová skupina:** Magisterské studium (Mgr.)

**Forma výuky:** Samostudium

**Časová dotace:** 100 minut

**Prostředí:** Google Colab / Python 3

---

## 📅 Doporučený harmonogram samostatné práce (100 minut)

* **00:00 – 00:20 (20 min):** Teoretický základ – Nashova rovnováha, Minimax a Vězňovo dilema
* **00:20 – 00:40 (20 min):** Praktická implementace: Výpočet Nashovy rovnováhy a Minimaxu v Pythonu
* **00:40 – 00:70 (30 min):** Evoluční simulátor (`simulation.py`) a experimenty s populacemi (Úkoly 1–3)
* **00:70 – 00:90 (20 min):** Úkol 4: Návrh učící se / adaptivní strategie (Q-Learning)
* **00:90 – 0100 (10 min):** Syntéza poznatků, analytické otázky a sebereflexe

---

## 1. Teoretický základ: Teorie her, Nashova rovnováha a Minimax

V multiagentních systémech se rozhodování agenta odvíjí od očekávaného chování ostatních subjektů. Matice odměn pro dva hráče určuje výnos $R_1(a_1, a_2)$ pro Hráče 1 a $R_2(a_1, a_2)$ pro Hráče 2 při volbě akcí $a_1, a_2 \in \{C, D\}$.

### Matice odměn Vězňova dilematu

Předpokládejme standardní konfiguraci odměn $(T, R, P, S)$, kde platí $T > R > P > S$ a zároveň $2R > T + S$:

* **$T$ (Temptation / Pokušení):** $5$ bodů za jednostrannou zradu
* **$R$ (Reward / Odměna):** $3$ body za oboustrannou spolupráci
* **$P$ (Punishment / Trest):** $1$ bod za oboustrannou zradu
* **$S$ (Sucker's payoff / Prohra):** $0$ bodů za podvedenou spolupráci

| Hráč 1 \ Hráč 2 | C (Spolupráce) | D (Zrada) |
| --- | --- | --- |
| **C (Spolupráce)** | **3 / 3** | **0 / 5** |
| **D (Zrada)** | **5 / 0** | **1 / 1** |

### Nashova rovnováha (Nash Equilibrium)

Profil akcí $(a_1^*, a_2^*)$ představuje **Nashovu rovnováhu**, pokud žádný z hráčů nemůže jednostrannou změnou své strategie dosáhnout vyššího výnosu:


$$\forall a_1: R_1(a_1^*, a_2^*) \ge R_1(a_1, a_2^*)$$

$$\forall a_2: R_2(a_1^*, a_2^*) \ge R_2(a_2^*, a_2)$$

V jednorázovém Vězňově dilematu je profil $(D, D)$ **jedinou ryzí Nashovou rovnováhou**, přestože profil $(C, C)$ přináší oběma vyšší užitek (Pareto optimalita).

### Principy rozhodování: Minimax

Algoritmus **Minimax** minimalizuje maximální možnou ztrátu (resp. maximalizuje garantovaný minimální zisk v nejhorším případě):


$$a_1^{\text{minimax}} = \arg\max_{a_1} \min_{a_2} R_1(a_1, a_2)$$

* Pro volbu **$C$**: Nejhorší možný výsledek nastane, když soupeř zvolí $D \rightarrow \text{zisk} = 0$.
* Pro volbu **$D$**: Nejhorší možný výsledek nastane, když soupeř zvolí $D \rightarrow \text{zisk} = 1$.
* **Minimax rozhodnutí:** Volba $D$, protože $\max(0, 1) = 1$.

---

## 2. Praktická ukázka: Nashova rovnováha a Minimax v Pythonu

V následující buňce si naimplementujeme nástroj pro analytické vyhodnocení matice odměn, nalezení Nashových rovnováh v ryzích strategiích a určení Minimax volby pro libovolnou hru $2 \times 2$.

In [ ]:
# [BUŇKA 1] Výpočet Nashovy rovnováhy a Minimaxu pro 2x2 hry
import numpy as np

def analyze_payoff_matrix(R1, R2, action_names=['C', 'D']):
    """
    Analýza matice odměn pro 2 hráče.
    R1: Matice odměn pro Hráče 1 (řádky = Hráč 1, sloupce = Hráč 2)
    R2: Matice odměn pro Hráče 2
    """
    num_actions = len(action_names)
    print("=== MATICE ODMĚN (Hráč 1 / Hráč 2) ===")
    for i in range(num_actions):
        row_str = [f"{R1[i,j]}/{R2[i,j]}" for j in range(num_actions)]
        print(f"{action_names[i]}: " + " | ".join(row_str))
    print("=" * 40)

    # 1. Výpočet Minimax volby pro Hráče 1
    worst_case_R1 = np.min(R1, axis=1) # nejhorší výsledek pro každou akci Hráče 1
    minimax_action_idx = np.argmax(worst_case_R1)
    print(f"🛡️ Minimax analýza pro Hráče 1:")
    for i, a in enumerate(action_names):
        print(f"   Akce '{a}': Garantované minimum = {worst_case_R1[i]} bodů")
    print(f"   -> Minimax volba: '{action_names[minimax_action_idx]}'\n")

    # 2. Hledání Nashových rovnováh v ryzích strategiích
    nash_equilibria = []
    for i in range(num_actions):
        for j in range(num_actions):
            # Je Hráč 1 v nejlepším stavu vůči volbě j Hráče 2?
            is_best_r1 = R1[i, j] == np.max(R1[:, j])
            # Je Hráč 2 v nejlepším stavu vůči volbě i Hráče 1?
            is_best_r2 = R2[i, j] == np.max(R2[i, :])

            if is_best_r1 and is_best_r2:
                nash_equilibria.append((action_names[i], action_names[j]))

    print(f"⚖️ Nalezené Nashovy rovnováhy (Hráč 1, Hráč 2):")
    if nash_equilibria:
        for eq in nash_equilibria:
            print(f"   -> Profil: ({eq[0]}, {eq[1]})")
    else:
        print("   -> Žádná ryzí Nashova rovnováha neexistuje.")

# Matice odměn Vězňova dilematu
R1_pd = np.array([[3, 0],
                  [5, 1]]) # Hráč 1
R2_pd = np.array([[3, 5],
                  [0, 1]]) # Hráč 2

analyze_payoff_matrix(R1_pd, R2_pd)

---

## 3. Evoluční simulátor dynamiky populací (`simulation.py`)

Představme si populaci jedinců, kteří mezi sebou opakovaně hrají Vězňovo dilema a na základě získaných bodů se rozmnožují.

* Může v takové simulaci existovat strategie, která přežije ve všech případech?
* Existuje skupina strategií, kterou nelze napadnout nově příchozí strategií (Evolučně stabilní strategie – ESS)?

### Princip simulátoru:

1. **Konfigurace:** Počáteční stav populace načítá soubor `config.json` (pokud neexistuje, vytvoří se výchozí s 10 instancemi od každé dostupné strategie).
2. **Generace a setkání:** V každé generaci odehraje každý jedinec $E$ náhodných zápasů (defaultně $E = 10$) proti náhodně vybraným oponentům. Zápas je iterované Vězňovo dilema o délce 200 kol.
3. **Ruletová selekce (Fitness Propagation):** Pravděpodobnost postupu/rozmnožení jedince do další generace je úměrná jeho kumulovanému skóre:

$$P(\text{jedinec}_i) = \frac{\text{Skóre}_i}{\sum_{j} \text{Skóre}_j}$$


4. **Vizualizace:** Výstupem je graf počtu jedinců a průměrného skóre v jednotlivých generacích.

In [ ]:
# [BUŇKA 2] Příprava prostředí a vytvoření modulů strategií
import os
import json

os.makedirs('strategies', exist_ok=True)
with open('strategies/__init__.py', 'w') as f: pass

# Definice základních strategií
strategies_code = {
"always_cooperate.py": """
class AlwaysCooperate:
    @staticmethod
    def author_name(): return "Pilat"
    @staticmethod
    def strategy_name(): return "Always Cooperate"
    def reset(self): pass
    def last_move(self, my, other): pass
    def play(self): return 'C'
def create_strategy(): return AlwaysCooperate()
""",
"always_defect.py": """
class AlwaysDefect:
    @staticmethod
    def author_name(): return "Pilat"
    @staticmethod
    def strategy_name(): return "Always Defect"
    def reset(self): pass
    def last_move(self, my, other): pass
    def play(self): return 'D'
def create_strategy(): return AlwaysDefect()
""",
"random_strat.py": """
import random
class RandomStrat:
    @staticmethod
    def author_name(): return "Pilat"
    @staticmethod
    def strategy_name(): return "Random"
    def reset(self): pass
    def last_move(self, my, other): pass
    def play(self): return 'C' if random.randint(0,1) == 0 else 'D'
def create_strategy(): return RandomStrat()
""",
"titfortat.py": """
class TitForTat:
    @staticmethod
    def author_name(): return "Pilat"
    @staticmethod
    def strategy_name(): return "Tit for Tat"
    def __init__(self): self.reset()
    def reset(self): self.last_op = None
    def last_move(self, my, other): self.last_op = other
    def play(self): return self.last_op if self.last_op else 'C'
def create_strategy(): return TitForTat()
"""
}

for filename, code in strategies_code.items():
    with open(f'strategies/{filename}', 'w') as f:
        f.write(code.strip())

print("✅ Základní moduly strategií byly uloženy do 'strategies/'.")

In [ ]:
# [BUŇKA 3] Implementace modulu simulation.py s podporou config.json a ruletové selekce
import collections
import copy
import importlib
import json
import os
import sys
import matplotlib.pyplot as plt
import numpy as np

# Herní matice odměn
REWARD_MATRIX = {
    ('C', 'C'): (3, 3),
    ('C', 'D'): (0, 5),
    ('D', 'C'): (5, 0),
    ('D', 'D'): (1, 1)
}

def play_game(s1, s2, iters=200):
    s1.reset()
    s2.reset()
    sc1, sc2 = 0, 0
    for _ in range(iters):
        m1, m2 = s1.play(), s2.play()
        r1, r2 = REWARD_MATRIX.get((m1, m2), (-1, -1))
        if r1 < 0 or r2 < 0:
            return 0, 0
        sc1 += r1
        sc2 += r2
        s1.last_move(m1, m2)
        s2.last_move(m2, m1)
    return sc1, sc2

def create_default_config(strat_modules, config_file='config.json'):
    config = {sm: 10 for sm in strat_modules}
    with open(config_file, 'w') as f:
        json.dump(config, f, indent=2)
    return config

def run_simulation(config_file='config.json', max_gen=10, encounters=10, custom_matrix=None):
    global REWARD_MATRIX
    if custom_matrix:
        REWARD_MATRIX = custom_matrix

    # Nacitani strategii
    strat_modules = []
    for f in os.listdir('strategies'):
        if f.endswith('.py') and not f.startswith('__'):
            strat_modules.append(f'strategies.{f[:-3]}')

    if not os.path.exists(config_file):
        config = create_default_config(strat_modules, config_file)
    else:
        with open(config_file) as f:
            config = json.load(f)

    pop = []
    strategies = {}
    for k, v in config.items():
        pop += v * [k]
        strategies[k] = importlib.import_module(k).create_strategy()

    pop_size = len(pop)
    str_counts = []
    str_scores = []

    for g in range(max_gen):
        scores = pop_size * [0]
        for s1_idx, s in enumerate(pop):
            s1 = strategies[s]
            for _ in range(encounters + 1):
                s2_idx = np.random.randint(0, pop_size)
                s2 = copy.deepcopy(strategies[pop[s2_idx]])
                sc1, sc2 = play_game(s1, s2, iters=200)
                scores[s1_idx] += sc1
                scores[s2_idx] += sc2

        counter = collections.Counter(pop)
        counts = {k: counter[k] for k in strategies.keys()}
        str_counts.append(counts)

        # Prumerne skore na strategii
        avg_scores = {}
        for k in strategies.keys():
            k_scores = [sc for sc, st in zip(scores, pop) if st == k]
            avg_scores[k] = np.mean(k_scores) if k_scores else 0
        str_scores.append(avg_scores)

        # Ruletova selekce (vyber s opakovanim)
        total_score = np.sum(scores)
        if total_score > 0:
            probs = np.array(scores) / total_score
        else:
            probs = np.ones(pop_size) / pop_size

        pop = list(np.random.choice(pop, size=pop_size, p=probs))

    # Vykresleni vysledku
    plt.figure(figsize=(12, 8))
    plt.subplot(2, 1, 1)
    for s in strategies.keys():
        plt.plot([sc[s] for sc in str_counts], label=strategies[s].strategy_name(), marker='o')
    plt.title("Vývoj počtu jedinců v populaci napříč generacemi")
    plt.ylabel("Počet jedinců")
    plt.grid(True)
    plt.legend()

    plt.subplot(2, 1, 2)
    for s in strategies.keys():
        plt.plot([sc[s] for sc in str_scores], label=strategies[s].strategy_name(), marker='s')
    plt.title("Průměrný zisk bodů na jedince v generaci")
    plt.xlabel("Generace")
    plt.ylabel("Průměrné skóre")
    plt.grid(True)
    plt.legend()

    plt.tight_layout()
    plt.show()

print("✅ Evoluční simulátor načten.")

---

## 4. Samostatné úkoly a experimenty

V této části budete provádět experimenty se simulátorem, měnit nastavení v `config.json` a upravovat herní matici odměn.

### 📝 Úkol 1: Směsi strategií – Invaze zrádců vs. Odolnost spolupráce

Otestujte dvě odlišné konfigurace populace a zasaďte výsledky do kontextu Nashovy rovnováhy a evolučně stabilní strategie (ESS):

1. **Scénář A:** Populace složená ze $45 \times$ `Always Cooperate` (AC) a $5 \times$ `Always Defect` (AD).
2. **Scénář B:** Populace složená ze $45 \times$ `Tit for Tat` (TFT) a $5 \times$ `Always Defect` (AD).

In [ ]:
# [BUŇKA 4] ÚKOL 1 – Testování invaze zrádců (AD) do AC vs. TFT

# --- SCÉNÁŘ A: 45x AC + 5x AD ---
config_a = {
    'strategies.always_cooperate': 45,
    'strategies.always_defect': 5
}
with open('config_scenario_a.json', 'w') as f:
    json.dump(config_a, f)

print("🧪 Spouštím Scénář A (45x AC + 5x AD)...")
run_simulation('config_scenario_a.json', max_gen=10)

# --- SCÉNÁŘ B: 45x TFT + 5x AD ---
config_b = {
    'strategies.titfortat': 45,
    'strategies.always_defect': 5
}
with open('config_scenario_b.json', 'w') as f:
    json.dump(config_b, f)

print("🧪 Spouštím Scénář B (45x TFT + 5x AD)...")
run_simulation('config_scenario_b.json', max_gen=10)

---

### 📝 Úkol 2: Nalezení prostředí pro přežití vlastní strategie

Vytvořte novou strategii `TitForTwoTats` (TF2T) a najděte dvě různé populace:

1. **Příznivou populaci:** Skupina strategií, kde `TF2T` přežije nebo zcela převládne.
2. **Nepříznivou populaci:** Skupina strategií, kde `TF2T` brzy vymře.

In [ ]:
# [BUŇKA 5] ÚKOL 2 – Návrh TF2T a vyhledání stabilního vs. destruktivního prostředí

tf2t_code = """
class TitForTwoTats:
    @staticmethod
    def author_name(): return "Student"
    @staticmethod
    def strategy_name(): return "Tit For Two Tats"
    def __init__(self): self.reset()
    def reset(self): self.defects = 0
    def last_move(self, my, other):
        if other == 'D': self.defects += 1
        else: self.defects = 0
    def play(self):
        return 'D' if self.defects >= 2 else 'C'
def create_strategy(): return TitForTwoTats()
"""
with open('strategies/titfortwotats.py', 'w') as f:
    f.write(tf2t_code.strip())

# TODO: Definujte příznivou a nepříznivou konfigurační populaci v json souborech
config_favorable = {
    'strategies.titfortwotats': 10,
    'strategies.always_cooperate': 20,
    'strategies.titfortat': 20
}
with open('config_fav.json', 'w') as f: json.dump(config_favorable, f)

print("🧪 Testuji Příznivé prostředí pro TF2T...")
run_simulation('config_fav.json', max_gen=10)

---

### 📝 Úkol 3: Změna matice odměn – Je hra stále Vězňovým dilematem?

Vyzkoušejte, co se stane s dynamikou populace, pokud změníte matici odměn tak, že porušíte základní nerovnosti Vězňova dilematu ($T > R > P > S$).

Otestujte např. matici hry **Stag Hunt (Lov na jelena)**, kde platí $R > T > P > S$:

* $(C, C) \rightarrow (4, 4)$
* $(C, D) \rightarrow (0, 3)$
* $(D, C) \rightarrow (3, 0)$
* $(D, D) \rightarrow (1, 1)$

In [ ]:
# [BUŇKA 6] ÚKOL 3 – Simulace s modifikovanou maticí odměn (Stag Hunt)

stag_hunt_matrix = {
    ('C', 'C'): (4, 4), # Odměna za společný lov jelena
    ('C', 'D'): (0, 3), # Hráč 1 utekl bez úlovku, Hráč 2 ulovil zajíce
    ('D', 'C'): (3, 0),
    ('D', 'D'): (1, 1)  # Obu ulovili malého zajíce
}

print("🧪 Spouštím simulaci s maticí Stag Hunt...")
run_simulation('config.json', max_gen=10, custom_matrix=stag_hunt_matrix)

---

### 📝 Úkol 4: Návrh učící se / adaptivní strategie (Q-Learning)

Naimplementujte strategii `QLearningAgent`, která nepoužívá pevně daná pravidla, ale učí se odhadovat očekávanou hodnotu ($Q$-hodnotu) pro akce $C$ a $D$ na základě předchozího tahu soupeře (stavy: `'START'`, `'OPP_C'`, `'OPP_D'`).

Aktualizační pravidlo Q-učení:


$$Q(s, a) \leftarrow Q(s, a) + \alpha \cdot \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$

In [ ]:
# [BUŇKA 7] ÚKOL 4 – Implementace Q-Learning Agenta pro IPD
q_learning_code = """
import random
import numpy as np

class QLearningAgent:
    @staticmethod
    def author_name(): return "Student AI"
    @staticmethod
    def strategy_name(): return "Q-Learning Agent"

    def __init__(self, alpha=0.1, gamma=0.9, epsilon=0.1):
        self.alpha = alpha     # Rychlost učení
        self.gamma = gamma     # Diskontní faktor
        self.epsilon = epsilon # Míra průzkumu (exploration)
        self.reset()

    def reset(self):
        # Stavy: 'START', 'C', 'D' (předchozí tah soupeře)
        # Akce: 0 = 'C', 1 = 'D'
        self.q_table = {
            'START': [3.0, 3.0],
            'C': [3.0, 3.0],
            'D': [1.0, 1.0]
        }
        self.current_state = 'START'
        self.last_action_idx = None

    def last_move(self, my_move, other_move):
        # Výpočet odměny z herní matice
        reward_map = {('C','C'):3, ('C','D'):0, ('D','C'):5, ('D','D'):1}
        r = reward_map.get((my_move, other_move), 0)

        next_state = other_move

        # Q-Learning update
        if self.last_action_idx is not None:
            old_q = self.q_table[self.current_state][self.last_action_idx]
            max_future_q = max(self.q_table[next_state])
            new_q = old_q + self.alpha * (r + self.gamma * max_future_q - old_q)
            self.q_table[self.current_state][self.last_action_idx] = new_q

        self.current_state = next_state

    def play(self):
        # Epsilon-greedy výběr akce
        if random.random() < self.epsilon:
            action_idx = random.choice([0, 1])
        else:
            q_vals = self.q_table[self.current_state]
            action_idx = np.argmax(q_vals)

        self.last_action_idx = action_idx
        return 'C' if action_idx == 0 else 'D'

def create_strategy(): return QLearningAgent()
"""

with open('strategies/q_learning.py', 'w') as f:
    f.write(q_learning_code.strip())

print("✅ Úkol 4: Q-Learning strategie vytvořena a uložena do 'strategies/q_learning.py'.")

---

## 5. Analytické otázky a Sebereflexe

Vyplňte odpovědi do textových polí níže:

### ❓ Analytické otázky

1. **Nashova rovnováha vs. Evoluční stabilita:**
* Vysvětlete na základě výsledků ze **Scénáře A** a **Scénáře B** v Úkolu 1, proč strategie `Always Cooperate` není evolučně stabilní (ESS), zatímco `Always Defect` a `Tit for Tat` dokážou invazi čelit.
* *Odpověď:*


2. **Aplikace Minimaxu v multiagentním prostředí:**
* Proč selhává Minimax algoritmus při snaze o dosažení globálně nejvyššího skóre v iterovaném Vězňově dilematu? S jakým předpokladem Minimax pracuje a proč tento předpoklad vede v IPD k podoptimálním výsledkům?
* *Odpověď:*


3. **Vliv modifikace matice odměn (Stag Hunt):**
* Jak se změnila dominanta strategií po úpravě matice v Úkolu 3 na odměny $R=4, T=3$? Byla volba zrady ($D$) stále dominantní akcí?
* *Odpověď:*



---

### 📝 Sebereflexe studenta

1. **Adaptabilita Q-Learning Agenta:**
* Jak se vámi naimplementovaný `QLearningAgent` vyvíjel během generací? Jaké parametry ($\alpha, \epsilon$) byste upravili, aby agent rychleji reagoval na převládající typ oponenta?
* *Odpověď:*


2. **Závěrečné zhodnocení:**
* Které prostředí (mix strategií) bylo pro vámi navržené agenty nejnáročnější a jaké hlavní poznatky si z dnešního laboratoře odnášíte?
* *Odpověď:*



---

> **Pokyny k odevzdání:** Uložte vyplněný notebook (`File -> Download -> Download .ipynb`) a odevzdejte jej společně se složkou vytvořených modulů strategií v archivu `.zip` do odevzdávárny v informačním systému.